# Fusion Testing — Speaker Separation via Timestamps

Concatenates `assistant_2.wav` + silence + `user_2.wav`, feeds the combined audio
to `albatrossv1-old.nemo`, and uses **word-level timestamps** to isolate only the
user portion of the transcript.

Runs the same experiment with both **CTC** and **RNNT** decoding strategies.

In [39]:
# ── Configuration ──────────────────────────────────────────────────────────────
NEMO_PATH          = "albatrossv1-old.nemo"   # EncDecHybridRNNTCTCBPEModel
ASSISTANT_WAV      = "assistant_2.wav"
USER_WAV           = "user_2.wav"
SILENCE_DURATION   = 0.5     # seconds of silence between the two clips
SAMPLE_RATE        = 16000   # expected sample rate; files will be resampled if needed
RNNT_BLANK_PENALTY = 0.0     # subtract from RNNT blank logit; 0.0 = off, try 1.0–3.0


In [40]:
import logging
import warnings
import numpy as np
import soundfile as sf
import torch

# Suppress noisy logs so notebook output is readable
for _ln in ("nemo_logger", "pytorch_lightning", "nemo.collections.asr", "nemo"):
    logging.getLogger(_ln).setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from omegaconf import OmegaConf, open_dict
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel

## 1. Load Model

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model on {device} ...")

model = EncDecHybridRNNTCTCBPEModel.restore_from(NEMO_PATH, map_location=device)
model.eval().to(device)

print(f"Model loaded")
print(f"  vocab_size        : {model.tokenizer.vocab_size}")
print(f"  sample_rate       : {model.cfg.preprocessor.sample_rate}")
print(f"  window_stride     : {model.cfg.preprocessor.window_stride * 1000:.0f} ms")
print(f"  subsampling_factor: {model.cfg.encoder.subsampling_factor}")


Loading model on cpu ...
Model loaded
  vocab_size        : 8192
  sample_rate       : 16000
  window_stride     : 10 ms
  subsampling_factor: 4


## 2. Load Audio and Build Combined Signal

In [42]:
def load_mono_f32(path: str, target_sr: int = SAMPLE_RATE) -> np.ndarray:
    """Load audio as float32 mono, resampling if needed."""
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)          # stereo -> mono
    if sr != target_sr:
        try:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        except ImportError:
            raise RuntimeError(
                f"Audio sample rate {sr} != {target_sr} and librosa is not installed."
            )
    return audio


assistant_audio = load_mono_f32(ASSISTANT_WAV)
user_audio      = load_mono_f32(USER_WAV)

assistant_dur_s = len(assistant_audio) / SAMPLE_RATE
user_dur_s      = len(user_audio)      / SAMPLE_RATE
silence_samples = int(SILENCE_DURATION * SAMPLE_RATE)
silence         = np.zeros(silence_samples, dtype=np.float32)

# Concatenate:  [assistant] + [silence] + [user]
combined = np.concatenate([assistant_audio, silence, user_audio])

# The user speech starts at this time offset in the combined signal
user_start_s = assistant_dur_s + SILENCE_DURATION

# Effective frame shift = preprocessor window_stride x encoder subsampling_factor
frame_shift_s = model.cfg.preprocessor.window_stride * model.cfg.encoder.subsampling_factor

print(f"assistant_2.wav : {assistant_dur_s:.3f}s  ({len(assistant_audio)} samples)")
print(f"silence         : {SILENCE_DURATION:.3f}s  ({silence_samples} samples)")
print(f"user_2.wav      : {user_dur_s:.3f}s  ({len(user_audio)} samples)")
print(f"combined        : {len(combined)/SAMPLE_RATE:.3f}s  ({len(combined)} samples)")
print(f"User speech starts at : {user_start_s:.3f}s")
print(f"frame_shift_s         : {frame_shift_s * 1000:.0f}ms  "
      f"({model.cfg.preprocessor.window_stride*1000:.0f}ms x {model.cfg.encoder.subsampling_factor})")


assistant_2.wav : 3.150s  (50400 samples)
silence         : 0.500s  (8000 samples)
user_2.wav      : 1.130s  (18080 samples)
combined        : 4.780s  (76480 samples)
User speech starts at : 3.650s
frame_shift_s         : 40ms  (10ms x 4)


## 3. Helper — Extract User Words from Timestamped Hypothesis

In [43]:
def extract_user_words(hypothesis, user_start_s: float, frame_shift_s: float):
    """
    Given a NeMo Hypothesis with word timestamps, return only the words
    that belong to the user portion of the combined audio.

    CTC timestamps can suffer from "smearing" across silence: a token that
    acoustically belongs to user speech may have its start_offset reported
    well before the boundary (CTC stretches tokens across silent regions).
    To handle this robustly we include a word when its END time crosses the
    boundary, not its start time.

    Parameters
    ----------
    hypothesis    : NeMo Hypothesis object (must have .timestamp['word'])
    user_start_s  : float — boundary time in seconds
    frame_shift_s : float — seconds per encoder frame

    Returns
    -------
    user_transcript : str   — user words joined by spaces
    all_words       : list  — all word dicts with word/start_s/end_s
    user_words      : list  — filtered subset
    """
    ts = hypothesis.timestamp
    if not isinstance(ts, dict) or 'word' not in ts:
        raise ValueError(
            "hypothesis.timestamp does not contain 'word' key. "
            "Make sure timestamps=True was passed to model.transcribe()."
        )

    all_words  = []
    user_words = []

    for w in ts['word']:
        entry = {
            "word"    : w["word"],
            "start_s" : w["start_offset"] * frame_shift_s,
            "end_s"   : w["end_offset"]   * frame_shift_s,
        }
        all_words.append(entry)
        # Use end_s as the filter criterion:
        # CTC smears tokens across silence, so start_s of a user word may
        # fall before the boundary. end_s is more reliable — it marks when
        # the token was actually decoded (i.e. when the audio arrived).
        if entry["end_s"] >= user_start_s:
            user_words.append(entry)

    user_transcript = " ".join(w["word"] for w in user_words).strip()
    return user_transcript, all_words, user_words


def print_word_table(words, label="", highlight_after_s=None):
    """Pretty-print a word timestamp table."""
    print(f"\n{chr(9472)*55}")
    if label:
        print(f"  {label}")
    print(f"  {'Word':<25} {'Start':>7}   {'End':>7}")
    print(f"  {chr(9472)*25} {chr(9472)*7}   {chr(9472)*7}")
    for w in words:
        # Highlight words whose END crosses the boundary (the user filter)
        marker = " ◀ USER" if (highlight_after_s is not None and w["end_s"] >= highlight_after_s) else ""
        print(f"  {w['word']:<25} {w['start_s']:>6.3f}s   {w['end_s']:>6.3f}s{marker}")
    print(f"{chr(9472)*55}")


In [46]:
from contextlib import contextmanager

@contextmanager
def rnnt_blank_penalty_ctx(model, penalty: float):
    """
    Temporarily subtract `penalty` from the RNNT blank logit at every
    joint step, discouraging blank emission and encouraging the decoder
    to emit more tokens per frame.

    NeMo does not expose blank_penalty natively for RNNT, so we
    monkey-patch joint_after_projection (same technique as _masked_joint
    in model.py). The patch is always restored on exit.

    penalty = 0.0  -> no-op (passthrough)
    penalty > 0.0  -> blank logit -= penalty  (e.g. 1.0–3.0)
    """
    if penalty == 0.0:
        yield
        return

    blank_id     = model.tokenizer.vocab_size   # blank is always the last logit
    original_fn  = model.joint.joint_after_projection

    def _penalized_joint(f, g):
        logits = original_fn(f, g)              # (B, T, U, vocab+1)
        logits[..., blank_id] -= penalty
        return logits

    model.joint.joint_after_projection = _penalized_joint
    try:
        yield
    finally:
        model.joint.joint_after_projection = original_fn


## 4. CTC Decoding with Word Timestamps

In [47]:
print("Switching to CTC decoder ...")
model.change_decoding_strategy(decoder_type="ctc")
print(f"cur_decoder: {model.cur_decoder}")

print("\nRunning CTC transcription on combined audio (timestamps=True) ...")
ctc_results = model.transcribe(
    audio=[combined],
    batch_size=1,
    return_hypotheses=True,
    timestamps=True,
    verbose=False,
)

# transcribe() returns List[Hypothesis] when return_hypotheses=True
ctc_hyp = ctc_results[0]

print(f"\nCTC full transcript : {ctc_hyp.text!r}")

Switching to CTC decoder ...
cur_decoder: ctc

Running CTC transcription on combined audio (timestamps=True) ...

CTC full transcript : 'can i confirm you would like a call back tomorrow at ten am ten am'


In [29]:
ctc_user_text, ctc_all_words, ctc_user_words = extract_user_words(
    ctc_hyp, user_start_s=user_start_s, frame_shift_s=frame_shift_s
)

print_word_table(ctc_all_words, label="CTC — all words", highlight_after_s=user_start_s)

print(f"\n{'═'*55}")
print(f"  CTC  |  Full transcript  : {ctc_hyp.text}")
print(f"  CTC  |  User transcript  : {ctc_user_text}")
print(f"  CTC  |  User word count  : {len(ctc_user_words)}")
print(f"  CTC  |  Boundary time    : {user_start_s:.3f}s")
print(f"{'═'*55}")


───────────────────────────────────────────────────────
  CTC — all words
  Word                        Start       End
  ───────────────────────── ───────   ───────
  can                        0.040s    0.080s
  i                          0.080s    0.199s
  confirm                    0.199s    0.359s
  you                        0.359s    0.757s
  would                      0.757s    0.876s
  like                       0.876s    1.036s
  a                          1.036s    1.195s
  call                       1.195s    1.354s
  back                       1.354s    1.554s
  tomorrow                   1.554s    1.753s
  at                         1.753s    2.111s
  ten                        2.111s    2.231s
  am                         2.231s    2.430s
  ten                        2.430s    4.222s ◀ USER
  am                         4.222s    4.461s ◀ USER
───────────────────────────────────────────────────────

═══════════════════════════════════════════════════════
  CTC  |  Full t

## 5. RNNT Decoding with Word Timestamps

In [62]:
print("Switching to RNNT decoder ...")
model.change_decoding_strategy(decoder_type="rnnt")
print(f"cur_decoder: {model.cur_decoder}")
RNNT_BLANK_PENALTY=2.5
print(f"RNNT_BLANK_PENALTY: {RNNT_BLANK_PENALTY}")

print("\nRunning RNNT transcription on combined audio (timestamps=True) ...")
with rnnt_blank_penalty_ctx(model, RNNT_BLANK_PENALTY):
    rnnt_results = model.transcribe(
        audio=[combined],
        batch_size=1,
        return_hypotheses=True,
        timestamps=True,
        verbose=False,
    )

rnnt_hyp = rnnt_results[0]

print(f"\nRNNT full transcript : {rnnt_hyp.text!r}")


Switching to RNNT decoder ...
cur_decoder: rnnt
RNNT_BLANK_PENALTY: 2.5

Running RNNT transcription on combined audio (timestamps=True) ...

========== BATCHED GREEDY DECODE START ==========
Batch size: 1
Encoder shape: (1, 120, 512)
Output lengths: [120]

--- NO PARTIAL HYPOTHESES PROVIDED ---

--- RUNNING DECODING COMPUTER ---
Decoding computer returned:
  batched_hyps: <class 'nemo.collections.asr.parts.utils.rnnt_utils.BatchedHyps'>
  alignments: present
  batched_state: <class 'nemo.collections.asr.parts.submodules.transducer_decoding.label_looping_base.BatchedLabelLoopingState'>

========== batched_hyps_to_hypotheses ==========
Requested batch_size: 1
Effective num_hyps: 1

--- RAW BATCHED OUTPUT ---
scores shape: torch.Size([1])
current_lengths: [15]
transcript shape: torch.Size([1, 1200])
timestamps shape: torch.Size([1, 1200])

[B0] score=-10.4843
[B0] current_length=15
[B0] raw transcript row: [330, 270, 674, 261, 980, 334, 283, 302, 547, 1746, 532, 615, 364, 615, 364, 8192, 

In [61]:
rnnt_user_text, rnnt_all_words, rnnt_user_words = extract_user_words(
    rnnt_hyp, user_start_s=user_start_s, frame_shift_s=frame_shift_s
)

print_word_table(rnnt_all_words, label="RNNT — all words", highlight_after_s=user_start_s)

print(f"\n{'═'*55}")
print(f"  RNNT |  Full transcript  : {rnnt_hyp.text}")
print(f"  RNNT |  User transcript  : {rnnt_user_text}")
print(f"  RNNT |  User word count  : {len(rnnt_user_words)}")
print(f"  RNNT |  Boundary time    : {user_start_s:.3f}s")
print(f"{'═'*55}")


───────────────────────────────────────────────────────
  RNNT — all words
  Word                        Start       End
  ───────────────────────── ───────   ───────
  can                        0.040s    0.080s
  i                          0.200s    0.240s
  confirm                    0.320s    0.360s
  you                        0.720s    0.760s
  would                      0.840s    0.880s
  like                       1.000s    1.040s
  to                         1.160s    1.200s
  call                       1.320s    1.360s
  back                       1.520s    1.560s
  tomorrow                   1.720s    1.760s
  at                         2.080s    2.120s
  ten                        2.240s    2.280s
  am                         2.400s    2.440s
  ten                        4.320s    4.360s ◀ USER
  am                         4.440s    4.480s ◀ USER
───────────────────────────────────────────────────────

═══════════════════════════════════════════════════════
  RNNT |  Full 

## 6. Side-by-Side Comparison

In [31]:
print("\n" + "═" * 70)
print("  COMPARISON SUMMARY")
print("═" * 70)
print(f"  Audio layout:")
print(f"    0.000s – {assistant_dur_s:.3f}s   assistant_2.wav")
print(f"    {assistant_dur_s:.3f}s – {user_start_s:.3f}s   silence ({SILENCE_DURATION:.3f}s)")
print(f"    {user_start_s:.3f}s – {len(combined)/SAMPLE_RATE:.3f}s   user_2.wav")
print(f"  User cutoff boundary  : {user_start_s:.3f}s")
print(f"  Frame shift           : {frame_shift_s * 1000:.0f}ms")
print()
print(f"  {'Decoder':<8}  {'Full transcript'}")
print(f"  {'─'*8}  {'─'*55}")
print(f"  {'CTC':<8}  {ctc_hyp.text}")
print(f"  {'RNNT':<8}  {rnnt_hyp.text}")
print()
print(f"  {'Decoder':<8}  {'User-only transcript  (words after boundary)'}")
print(f"  {'─'*8}  {'─'*55}")
print(f"  {'CTC':<8}  {ctc_user_text or '(no words found after boundary)'}")
print(f"  {'RNNT':<8}  {rnnt_user_text or '(no words found after boundary)'}")
print("═" * 70)


══════════════════════════════════════════════════════════════════════
  COMPARISON SUMMARY
══════════════════════════════════════════════════════════════════════
  Audio layout:
    0.000s – 3.150s   assistant_2.wav
    3.150s – 3.650s   silence (0.500s)
    3.650s – 4.780s   user_2.wav
  User cutoff boundary  : 3.650s
  Frame shift           : 40ms

  Decoder   Full transcript
  ────────  ───────────────────────────────────────────────────────
  CTC       can i confirm you would like a call back tomorrow at ten am ten am
  RNNT      can i confirm you would like a call back tomorrow at ten am

  Decoder   User-only transcript  (words after boundary)
  ────────  ───────────────────────────────────────────────────────
  CTC       ten am
  RNNT      (no words found after boundary)
══════════════════════════════════════════════════════════════════════


## 7. Experiment — Vary Silence Duration

Re-run with different silence gaps to see how timing shifts affect the boundary cut.

In [ ]:
# Switch back to CTC for the experiment (change to 'rnnt' if preferred)
model.change_decoding_strategy(decoder_type="ctc")

silence_durations = [0.0, 0.25, 0.5, 1.0]

print(f"\n{'Silence':>8}  {'Boundary':>10}  {'User transcript'}")
print(f"{'─'*8}  {'─'*10}  {'─'*50}")

for sil in silence_durations:
    sil_samples    = int(sil * SAMPLE_RATE)
    combined_exp   = np.concatenate([assistant_audio, np.zeros(sil_samples, np.float32), user_audio])
    boundary_exp   = assistant_dur_s + sil

    res = model.transcribe(
        audio=[combined_exp],
        batch_size=1,
        return_hypotheses=True,
        timestamps=True,
        verbose=False,
    )
    hyp_exp = res[0]   # List[Hypothesis] — take first item
    user_txt, _, _ = extract_user_words(hyp_exp, boundary_exp, frame_shift_s)
    print(f"{sil:>7.2f}s  {boundary_exp:>9.3f}s  {user_txt or '(none)'}")